##  Final Evaluation on Held-Out Test Sets (Real-World Performance)

In this step, we perform a comprehensive evaluation of our final **Logistic Regression models** on the **held-out test datasets** (20% split, never seen during training). This allows us to estimate **real-world performance** using:

-  **Confusion Matrix** — Actual vs Predicted breakdown  
-  **Classification Report** — Precision, Recall, F1 Score, Accuracy  
-  **ROC-AUC Score** — Model's ability to rank positives over negatives  
-  **Probability-based predictions** (used for threshold tuning later)

These evaluations confirm the effectiveness and generalization of each model under realistic, non-resampled distributions. All reports and plots are saved for documentation and comparison.


In [1]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

part17_metrics_dir = os.path.join(METRICS_DIR, "part17_final_metrics")
os.makedirs(part17_metrics_dir, exist_ok=True)

part17_plots_dir = os.path.join(PLOTS_DIR, "part17_final_metrics")
os.makedirs(part17_plots_dir, exist_ok=True)



In [2]:
import pandas as pd
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

def evaluate_final_model(model_file, test_csv, target_col, label):
    # Classification report → metrics
    report_path = os.path.join(part17_metrics_dir, f"classification_report_{label}.csv")

    # Confusion matrix → plots
    cm_path = os.path.join(part17_plots_dir, f"conf_matrix_{label}.png")

    # Load model and test data
    model = joblib.load(model_file)
    df = pd.read_csv(test_csv)
    df = df[df[target_col].isin(["Yes", "No"])]

    X_test = df.drop(columns=[target_col])
    y_true = df[target_col].map({"No": 0, "Yes": 1})

    # Predict labels and probabilities
    y_pred = model.predict(X_test)
    y_pred = pd.Series(y_pred).map({"No": 0, "Yes": 1})
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    auc_score = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)

    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(report_path)
    print(f" Saved classification report to {report_path}")

    # Plot confusion matrix
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Predicted: No", "Predicted: Yes"],
                yticklabels=["Actual: No", "Actual: Yes"])
    plt.title(f"Confusion Matrix - {label}")
    plt.tight_layout()
    plt.savefig(cm_path)
    plt.close()
    print(f" Saved confusion matrix to {cm_path}")

    # Print metrics summary
    print(f"\n Final Evaluation: {label}")
    print(f"ROC-AUC: {auc_score:.4f}")
    print(f"Recall (Yes): {report['1']['recall']:.4f}")
    print(f"Precision (Yes): {report['1']['precision']:.4f}")
    print(f"F1 Score (Yes): {report['1']['f1-score']:.4f}")
    print(f"Accuracy: {report['accuracy']:.4f}")


In [3]:
# === Run for all 3 models ===
evaluate_final_model(
    model_file=os.path.join(METRICS_DIR, "part14_final_models", "model_highbp.pkl"),
    test_csv=os.path.join(STATS_DIR, "part14_final_models", "test_highbp.csv"),
    target_col="Has a high blood pressure",
    label="HighBP"
)

evaluate_final_model(
    model_file=os.path.join(METRICS_DIR, "part14_final_models", "model_diabetes.pkl"),
    test_csv=os.path.join(STATS_DIR, "part14_final_models", "test_diabetes.csv"),
    target_col="Has diabetes",
    label="Diabetes"
)

evaluate_final_model(
    model_file=os.path.join(METRICS_DIR, "part14_final_models", "model_cardio.pkl"),
    test_csv=os.path.join(STATS_DIR, "part14_final_models", "test_cardio.csv"),
    target_col="Cardiovascular condition (Heart disease or stroke)",
    label="Cardio"
)


 Saved classification report to d:\Projects\health-risk-prediction\outputs\metrics\part17_final_metrics\classification_report_HighBP.csv
 Saved confusion matrix to d:\Projects\health-risk-prediction\outputs\plots\part17_final_metrics\conf_matrix_HighBP.png

 Final Evaluation: HighBP
ROC-AUC: 0.8095
Recall (Yes): 0.7848
Precision (Yes): 0.5060
F1 Score (Yes): 0.6153
Accuracy: 0.7173
 Saved classification report to d:\Projects\health-risk-prediction\outputs\metrics\part17_final_metrics\classification_report_Diabetes.csv
 Saved confusion matrix to d:\Projects\health-risk-prediction\outputs\plots\part17_final_metrics\conf_matrix_Diabetes.png

 Final Evaluation: Diabetes
ROC-AUC: 0.8524
Recall (Yes): 0.8025
Precision (Yes): 0.2576
F1 Score (Yes): 0.3901
Accuracy: 0.7513
 Saved classification report to d:\Projects\health-risk-prediction\outputs\metrics\part17_final_metrics\classification_report_Cardio.csv
 Saved confusion matrix to d:\Projects\health-risk-prediction\outputs\plots\part17_fina